# Initial Clustering

## Unzip Data + Prepare Data
Raw -> Text Tokens + P + I + O

In [32]:
import tarfile
import os
import spacy
import numpy as np
from pathlib import Path
from itertools import chain
# ONLY RUN THIS ONCE TO EXTRACT DATA FOR THE FIRST TIME
#path_tofile = "./data/ebm_nlp_2_00.tar.gz"
#extract_directory = os.path.dirname(path_tofile)
#
#if tarfile.is_tarfile(path_tofile):
#    with tarfile.open(path_tofile) as f:
#        f.extractall(path=extract_directory)

In [33]:
DATA_DIR = Path("./data/ebm_nlp_2_00")

In [34]:
def get_doc_ids(split="train", label_type="participants"):
    """ 
    split: 'train' or 'test' 
    """

    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type  # assuming that the split is the same for all entity types, we can just look at one of them
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
    return sorted(doc_ids)

In [35]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

In [36]:
def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test' 
    """
    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

In [37]:
def hierarchical_to_bio(tags):
    """
    Convert EBM-NLP hierarchical labels (0–4) to flat BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B", "I").
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")

        prev = t
        

    return bio

def convert_all_labels_to_bio(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_bio(doc_labels)
    return labels

In [59]:
def hierarchical_to_binary(tags):
    """
    Convert EBM-NLP hierarchical labels (0–4) to flat binary masks (0 or 1).
    0 means 'Not the entity' (O).
    Anything > 0 means 'Is the entity' (B or I).
    """
    binary_mask = []
    for t in tags:
        t = int(t)
        if t == 0:
            binary_mask.append(0)
        else:
            binary_mask.append(1)
            
    return binary_mask

def convert_all_labels_to_binary(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_binary(doc_labels)
    return labels

## Load the Data

In [60]:
p_ids = set(get_doc_ids("train", "participants"))
i_ids = set(get_doc_ids("train", "interventions"))
o_ids = set(get_doc_ids("train", "outcomes"))
train_ids = sorted(list(p_ids & i_ids & o_ids))

p_test_ids = set(get_doc_ids("test", "participants"))
i_test_ids = set(get_doc_ids("test", "interventions"))
o_test_ids = set(get_doc_ids("test", "outcomes"))
test_ids = sorted(list(p_test_ids & i_test_ids & o_test_ids))

train_tokens = load_documents(train_ids)
test_tokens = load_documents(test_ids)

train_labels_p = load_labels(train_ids, "participants", "train")
train_labels_i = load_labels(train_ids, "interventions", "train")
train_labels_o = load_labels(train_ids, "outcomes", "train")

test_labels_p = load_labels(test_ids, "participants", "test")
test_labels_i = load_labels(test_ids, "interventions", "test")
test_labels_o = load_labels(test_ids, "outcomes", "test")

# train_labels_p = convert_all_labels_to_bio(train_labels_p)
# test_labels_p = convert_all_labels_to_bio(test_labels_p)

# train_labels_i = convert_all_labels_to_bio(train_labels_i)
# test_labels_i = convert_all_labels_to_bio(test_labels_i)

# train_labels_o = convert_all_labels_to_bio(train_labels_o)
# test_labels_o = convert_all_labels_to_bio(test_labels_o)

train_labels_p = convert_all_labels_to_binary(train_labels_p)
test_labels_p = convert_all_labels_to_binary(test_labels_p)

train_labels_i = convert_all_labels_to_binary(train_labels_i)
test_labels_i = convert_all_labels_to_binary(test_labels_i)

train_labels_o = convert_all_labels_to_binary(train_labels_o)
test_labels_o = convert_all_labels_to_binary(test_labels_o)

In [48]:
import shutil
import spacy
import numpy as np
from pathlib import Path
from itertools import chain
from tqdm import tqdm

nlp = spacy.load("en_core_web_md")
processed_dir = Path("./data/ebm_nlp_2_00/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

def vectorize_and_label_sentences(tokens_nested, labels_p_nested, labels_i_nested, labels_o_nested, split_name="Data"):
    sent_vectors = []
    sent_labels_p = []
    sent_labels_i = []
    sent_labels_o = []
    
    sent_texts = []

    print(f"Processing {split_name} into sentences...")
    for tokens, lp, li, lo in tqdm(zip(tokens_nested, labels_p_nested, labels_i_nested, labels_o_nested), 
                                    total=len(tokens_nested)):
        
        doc = nlp(" ".join(tokens))
        
        for sent in doc.sents:
            if len(sent.text.split()) < 3: 
                continue
            
            sent_vectors.append(sent.vector)
            sent_texts.append(sent.text)
            
            def has_entity(label_list_doc, start_idx, end_idx):
                segment = label_list_doc[start_idx:end_idx]
                return 1 if any(lab != 'O' for lab in segment) else 0

            sent_labels_p.append(has_entity(lp, sent.start, sent.end))
            sent_labels_i.append(has_entity(li, sent.start, sent.end))
            sent_labels_o.append(has_entity(lo, sent.start, sent.end))

    return np.array(sent_vectors), \
           np.array(sent_labels_p), np.array(sent_labels_i), np.array(sent_labels_o), \
           sent_texts

X_train_sent, y_p_train_sent, y_i_train_sent, y_o_train_sent, train_sent_texts = \
    vectorize_and_label_sentences(train_tokens, train_labels_p, train_labels_i, train_labels_o, "Train Split")

X_test_sent, y_p_test_sent, y_i_test_sent, y_o_test_sent, test_sent_texts = \
    vectorize_and_label_sentences(test_tokens, test_labels_p, test_labels_i, test_labels_o, "Test Split")

np.savez_compressed(
    processed_dir / "ebm_sentence_vectors.npz", 
    X_train=X_train_sent, y_p_train=y_p_train_sent, y_i_train=y_i_train_sent, y_o_train=y_o_train_sent,
    X_test=X_test_sent, y_p_test=y_p_test_sent, y_i_test=y_i_test_sent, y_o_test=y_o_test_sent,
    texts_train=train_sent_texts,
    texts_test=test_sent_texts
)

print(f"Sentence data saved to {processed_dir}")

Processing Train Split into sentences...


100%|███████████████████████████████████████| 4457/4457 [02:21<00:00, 31.49it/s]


Processing Test Split into sentences...


100%|█████████████████████████████████████████| 184/184 [00:05<00:00, 32.44it/s]


Sentence data saved to data/ebm_nlp_2_00/processed


# Prepare the Data for LLM

In [61]:
import numpy as np
from pathlib import Path

processed_dir = Path("./data/ebm_nlp_2_00/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_texts = [" ".join(tokens) for tokens in train_tokens]
test_texts = [" ".join(tokens) for tokens in test_tokens]

output_file = processed_dir / "ebm_abstracts_full.npz"

np.savez_compressed(
    output_file,
    
    # Train Data
    train_texts=np.array(train_texts, dtype=object),
    train_p=np.array(train_labels_p, dtype=object),
    train_i=np.array(train_labels_i, dtype=object),
    train_o=np.array(train_labels_o, dtype=object),
    
    # train Data
    test_texts=np.array(test_texts, dtype=object),
    test_p=np.array(test_labels_p, dtype=object),
    test_i=np.array(test_labels_i, dtype=object),
    test_o=np.array(test_labels_o, dtype=object)
)

print(f"Successfully saved full abstracts and labels to {output_file}")

Successfully saved full abstracts and labels to data/ebm_nlp_2_00/processed/ebm_abstracts_full.npz


# Example Read

In [62]:
import numpy as np

data = np.load('./data/ebm_nlp_2_00/processed/ebm_abstracts_full.npz', allow_pickle=True)

train_texts = data['train_texts']
train_p = data['train_p']
train_i = data['train_i']
train_o = data['train_o']

first_abstract = train_texts[0]
first_p_labels = train_p[0]
first_i_labels = train_i[0]
first_o_labels = train_o[0]

print("=== FIRST ABSTRACT TEXT ===")
print(first_abstract)

print("\n=== POPULATION (P) LABELS ===")
print(first_p_labels)

print("\n=== INTERVENTION (I) LABELS ===")
print(first_i_labels)

print("\n=== OUTCOME (O) LABELS ===")
print(first_o_labels)

=== FIRST ABSTRACT TEXT ===
[ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] . Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection . To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarithromycin ( CAM ) and metrinidazole ( MTZ ) . The drug combination and administration periods in the PPI group were LPZ 30 mg , CAM 400 mg , MTZ 500 mg ( LCM group ) . The ranitidine group received RNT 300 mg , CAM 400 mg , MTZ 500 mg ( RCM group ) . The cure rate of H. pylori infection was 88 % in the LCM group ; 95 % CI 79-97 and 92 % in the RCM group ; 95 % CI 84-99 .

=== POPULATION (P) LABELS ===
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0